In [4]:
from dotenv import load_dotenv
load_dotenv("../Lesson1/.env")

True

In [5]:
from openai import OpenAI
import os

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print(os.getenv("BASE_URL"))

https://models.inference.ai.azure.com


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
v1.dot(dv)

np.float32(0.32332397)

In [10]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [11]:
v1.dot(v2)


np.float32(-0.14271772)

In [12]:
v2.dot(dv)  


np.float32(0.019730438)

In [13]:
import sys
import os

# Adds the parent folder to Python's search path
sys.path.insert(0, os.path.abspath(".."))

# Try your import again
from Lesson1.ingest import load_faq_data

documents = load_faq_data()


In [14]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [15]:
from tqdm.auto import tqdm

In [16]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1342

In [17]:
import numpy as np
X = np.array(vectors)

In [18]:
X.shape

(1342, 384)

In [19]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)

In [21]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [22]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [23]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [24]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [27]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 536, 899, 617,   2])

In [26]:
top5 = top5[::-1]
top5

array([  2, 617, 899, 536,   7])

In [28]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date